# A03 - 二维通道对流扩散传热

本例先不求流场，而是给定层流通道速度剖面：

`u(y)=4*y*(1-y), v=0`

求温度场 `T(x,y)`：

`u*T_x + v*T_y = (1/Pe)*(T_xx + T_yy)`

边界：入口 `T(0,y)=1`，上下壁 `T(x,0)=T(x,1)=0`，出口采用近似零轴向梯度 `T_x(1,y)=0`。

这是流体传热 PINN 的最小完整框架：对流项、扩散项、入口/壁面/出口边界都出现了。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(103)
device = "cuda" if torch.cuda.is_available() else "cpu"
Pe = 20.0
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, width=64, depth=5):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, xy):
        return self.net(xy)

def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

def velocity(y):
    return 4 * y * (1 - y), torch.zeros_like(y)

model = MLP().to(device)

In [ ]:
def heat_residual(x, y):
    x.requires_grad_(True)
    y.requires_grad_(True)
    T = model(torch.cat([x, y], dim=1))
    T_x, T_y = grad(T, x), grad(T, y)
    T_xx, T_yy = grad(T_x, x), grad(T_y, y)
    u, v = velocity(y)
    return u * T_x + v * T_y - (T_xx + T_yy) / Pe

def boundary_loss(n=160):
    s = torch.rand(n, 1, device=device)
    x_in, y_in = torch.zeros_like(s), s
    T_in = model(torch.cat([x_in, y_in], dim=1))
    loss_in = torch.mean((T_in - 1.0)**2)

    x_wall = torch.rand(n, 1, device=device)
    y0 = torch.zeros_like(x_wall)
    y1 = torch.ones_like(x_wall)
    T_wall0 = model(torch.cat([x_wall, y0], dim=1))
    T_wall1 = model(torch.cat([x_wall, y1], dim=1))
    loss_wall = torch.mean(T_wall0**2) + torch.mean(T_wall1**2)

    y_out = torch.rand(n, 1, device=device)
    x_out = torch.ones_like(y_out)
    x_out.requires_grad_(True)
    T_out = model(torch.cat([x_out, y_out], dim=1))
    T_x_out = grad(T_out, x_out)
    loss_out = torch.mean(T_x_out**2)
    return loss_in + loss_wall + 0.2 * loss_out

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(9000):
    x = torch.rand(1024, 1, device=device)
    y = torch.rand(1024, 1, device=device)
    loss_pde = torch.mean(heat_residual(x, y)**2)
    loss_bc = boundary_loss()
    loss = loss_pde + 10 * loss_bc
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    history.append([loss.item(), loss_pde.item(), loss_bc.item()])
    if epoch % 1500 == 0:
        print(f"epoch={epoch:4d}, total={loss.item():.3e}, pde={loss_pde.item():.3e}, bc={loss_bc.item():.3e}")

In [ ]:
nx, ny = 120, 60
xs = torch.linspace(0, 1, nx, device=device)
ys = torch.linspace(0, 1, ny, device=device)
X, Y = torch.meshgrid(xs, ys, indexing="ij")
xy = torch.cat([X.reshape(-1,1), Y.reshape(-1,1)], dim=1)
with torch.no_grad():
    T = model(xy).reshape(nx, ny)

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.imshow(T.cpu().T, origin="lower", extent=[0,1,0,1], aspect="auto")
plt.colorbar(); plt.title("temperature field")
plt.xlabel("x"); plt.ylabel("y")
plt.subplot(1, 2, 2)
plt.semilogy(torch.tensor(history)[:,0], label="total")
plt.semilogy(torch.tensor(history)[:,1], label="pde")
plt.semilogy(torch.tensor(history)[:,2], label="bc")
plt.legend(); plt.title("loss")
plt.tight_layout(); plt.show()